# Examora — public demo link

Run the three cells in order. The last one prints a public URL anyone can open.

The link stays alive while this notebook is running.

In [ ]:
#@title 1. Get the code and install everything (about 3 minutes)

REPO = "https://github.com/Farah296-coder/EXAMORA.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

!rm -rf examora
!git clone --branch $BRANCH --depth 1 $REPO examora
%cd examora

!pip install -q -r requirements.txt
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

print("ready")

In [ ]:
#@title 2. Your Groq key

import os

GROQ_API_KEY = ""  #@param {type:"string"}
GROQ_MODEL = "openai/gpt-oss-120b"  #@param ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.8-27b"]

os.environ["GROQ_API_KEY"] = GROQ_API_KEY.strip()
os.environ["GROQ_MODEL"] = GROQ_MODEL

print("key set:", bool(os.environ["GROQ_API_KEY"]), "| model:", GROQ_MODEL)

In [ ]:
#@title 3. Start the app and print the public link

import re
import subprocess
import time
import urllib.request

subprocess.Popen(
    ["python", "-m", "uvicorn", "api:app", "--app-dir", "src", "--port", "8000"],
    stdout=open("api.log", "w"), stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        if urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2).status == 200:
            print("backend is up")
            break
    except Exception:
        time.sleep(2)
else:
    print("backend did not start — check api.log")

subprocess.Popen(
    ["python", "-m", "streamlit", "run", "src/examora_chat.py",
     "--server.port", "8501", "--server.address", "0.0.0.0", "--server.headless", "true"],
    stdout=open("app.log", "w"), stderr=subprocess.STDOUT,
)
time.sleep(8)

subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
    stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(40):
    time.sleep(3)
    try:
        found = re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
        if found:
            public_url = found[0]
            break
    except FileNotFoundError:
        pass

if public_url:
    print("\n" + "=" * 60)
    print("  Share this link:", public_url)
    print("  It works while this notebook keeps running.")
    print("=" * 60)
else:
    print("no link yet — run this cell again, or check tunnel.log")